# District → State Crop Entropy

Loads the per-district Shannon entropy CSVs exported by `jordi_crop_diversity.js` (GEE) and rolls them up to a state-level entropy in `crop_entropy_by_state.csv`.

Two export formats are handled, based on what columns are actually present:

- **Exact method** — if the CSV has a `class_histogram` column (`"1:120,2:450,..."` per district, produced by the current `jordi_crop_diversity.js`), district histograms are summed into one state-wide histogram and entropy is computed from those combined counts. Mathematically identical to scanning the whole state's pixels at once.
- **Approximate fallback** — the sample file (`mp_CropEntropy_District.csv`) only has `crop_entropy_district` (no histogram — it's from an older export). In that case state entropy is approximated as the **area-weighted average** of district entropies, using each district's polygon in the `.geo` column as the weight. This isn't mathematically identical to the exact method (entropy doesn't average linearly), but it's a reasonable stand-in when only per-district entropy values are available. Re-export with the current `jordi_crop_diversity.js` to get the exact number.

**Before running:**
1. In the GEE Code Editor, run `jordi_crop_diversity.js` and, in the Tasks tab, run every `<state_key>_CropEntropy_District` export task (CSV → Google Drive).
2. Download those CSVs from Drive into the folder set as `DISTRICT_CSV_DIR` below (default: `data/crop_entropy_district/`).
3. `pip install pandas numpy shapely` (shapely is only needed for the approximate fallback's area weighting).

In [2]:
import glob
import json
import os

import numpy as np
import pandas as pd

# Folder containing the downloaded '<state_key>_CropEntropy_District.csv' files.
DISTRICT_CSV_DIR = '../../data/crop_entropy'

csv_paths = sorted(glob.glob(os.path.join(DISTRICT_CSV_DIR, '*_CropEntropy_District.csv')))
print(f"Found {len(csv_paths)} district CSVs in {DISTRICT_CSV_DIR}")
csv_paths

Found 34 district CSVs in ../../data/crop_entropy


['../../data/crop_entropy/andaman_nicobar_CropEntropy_District.csv',
 '../../data/crop_entropy/andhra_pradesh_CropEntropy_District.csv',
 '../../data/crop_entropy/arunachal_pradesh_CropEntropy_District.csv',
 '../../data/crop_entropy/assam_CropEntropy_District.csv',
 '../../data/crop_entropy/bihar_CropEntropy_District.csv',
 '../../data/crop_entropy/chandigarh_CropEntropy_District.csv',
 '../../data/crop_entropy/chhattisgarh_CropEntropy_District.csv',
 '../../data/crop_entropy/dadra_nagar_haveli_CropEntropy_District.csv',
 '../../data/crop_entropy/delhi_CropEntropy_District.csv',
 '../../data/crop_entropy/goa_CropEntropy_District.csv',
 '../../data/crop_entropy/gujarat_CropEntropy_District.csv',
 '../../data/crop_entropy/haryana_CropEntropy_District.csv',
 '../../data/crop_entropy/himachal_pradesh_CropEntropy_District.csv',
 '../../data/crop_entropy/jharkhand_CropEntropy_District.csv',
 '../../data/crop_entropy/karnataka_CropEntropy_District.csv',
 '../../data/crop_entropy/kerala_CropE

## Helpers

`class_histogram`, when present, is a string like `"1:120,2:450,3:80"` (GEE dictionaries don't serialize cleanly to CSV, so `jordi_crop_diversity.js` encodes it this way). Summing these across every district in a state gives the state's total per-class pixel counts.

When it's absent, `district_area` parses the `.geo` GeoJSON polygon string GEE attaches to each exported feature, so districts can be weighted by their (planar, in degrees²) area rather than counted equally — a district twice the size of another should count twice as much toward the state average.

In [3]:
def parse_histogram(hist_str):
    """'1:120,2:450,3:80' -> {'1': 120.0, '2': 450.0, '3': 80.0}"""
    counts = {}
    if not isinstance(hist_str, str) or hist_str.strip() == '':
        return counts
    for pair in hist_str.split(','):
        if ':' not in pair:
            continue
        key, val = pair.split(':', 1)
        key = key.strip()
        try:
            counts[key] = counts.get(key, 0.0) + float(val)
        except ValueError:
            continue
    return counts


def shannon_entropy(counts):
    """-sum(p * ln(p)) over class counts. NaN if there are no counts."""
    if not counts:
        return np.nan
    values = np.array(list(counts.values()), dtype=float)
    total = values.sum()
    if total <= 0:
        return np.nan
    p = values / total
    return float(-(p * np.log(p)).sum())


def district_area(geo_str):
    """Planar area (in degrees^2) of a district's GeoJSON geometry string, as
    exported in the '.geo' column. Only used as a relative weight between
    districts within the same state, so the lack of a proper projection
    doesn't matter -- it's not being used as a real-world area."""
    try:
        from shapely.geometry import shape
        return shape(json.loads(geo_str)).area
    except Exception:
        return np.nan

In [4]:
def state_entropy_from_district_csv(csv_path):
    df = pd.read_csv(csv_path)
    n_districts = len(df)
    district_avg_entropy = df['crop_entropy_district'].mean()

    has_histogram = 'class_histogram' in df.columns and df['class_histogram'].notna().any()

    if has_histogram:
        # ── Exact: sum per-district class counts into one state histogram ──
        state_counts = {}
        for hist_str in df['class_histogram'].dropna():
            for cls, count in parse_histogram(hist_str).items():
                state_counts[cls] = state_counts.get(cls, 0.0) + count
        state_entropy = shannon_entropy(state_counts)
        method = 'exact (summed class histograms)'
    else:
        # ── Approximate fallback: area-weighted average of district entropies ──
        # Used when the CSV predates the class_histogram export (like the
        # sample mp_CropEntropy_District.csv). Not mathematically identical to
        # the exact method -- entropy doesn't average linearly across regions
        # -- but a reasonable stand-in given only per-district entropy values.
        if '.geo' in df.columns:
            weights = df['.geo'].apply(district_area)
        else:
            weights = pd.Series(1.0, index=df.index)  # equal weights if no geometry either
        weights = weights.fillna(0)
        if weights.sum() == 0:
            weights = pd.Series(1.0, index=df.index)  # avoid divide-by-zero; fall back to equal weights
        weights = weights / weights.sum()
        state_entropy = float((df['crop_entropy_district'] * weights).sum())
        method = 'approximate (area-weighted average of district entropies)'

    return state_entropy, district_avg_entropy, n_districts, method

## Run over every state's district CSV

In [5]:
rows = []

for csv_path in csv_paths:
    state_key = os.path.basename(csv_path).replace('_CropEntropy_District.csv', '')
    state_entropy, district_avg_entropy, n_districts, method = state_entropy_from_district_csv(csv_path)

    rows.append({
        'state': state_key,
        'crop_entropy_state': state_entropy,
        'crop_entropy_district_avg': district_avg_entropy,
        'n_districts': n_districts,
        'method': method,
    })
    print(f"{state_key:>20s}: state = {state_entropy:.4f}   district avg = {district_avg_entropy:.4f}   (n={n_districts}, {method})")

state_df = pd.DataFrame(rows)
state_df

     andaman_nicobar: state = 0.0000   district avg = nan   (n=3, approximate (area-weighted average of district entropies))
      andhra_pradesh: state = 0.5581   district avg = 0.6103   (n=13, approximate (area-weighted average of district entropies))
   arunachal_pradesh: state = 0.0457   district avg = 0.0583   (n=8, approximate (area-weighted average of district entropies))
               assam: state = 0.5767   district avg = 0.6368   (n=33, approximate (area-weighted average of district entropies))
               bihar: state = 0.9463   district avg = 0.9420   (n=38, approximate (area-weighted average of district entropies))
          chandigarh: state = 0.1246   district avg = 0.1246   (n=1, approximate (area-weighted average of district entropies))
        chhattisgarh: state = 0.7754   district avg = 0.7700   (n=28, approximate (area-weighted average of district entropies))
  dadra_nagar_haveli: state = 0.0280   district avg = 0.0343   (n=3, approximate (area-weighted average

,state,crop_entropy_state,crop_entropy_district_avg,n_districts,method
0,andaman_nicobar,0.000000,NaN,3,approximate (area-weighted average of district...
1,andhra_pradesh,0.558144,0.610298,13,approximate (area-weighted average of district...
2,arunachal_pradesh,0.045728,0.058318,8,approximate (area-weighted average of district...
3,assam,0.576714,0.636825,33,approximate (area-weighted average of district...
4,bihar,0.946264,0.941960,38,approximate (area-weighted average of district...
5,chandigarh,0.124584,0.124584,1,approximate (area-weighted average of district...
6,chhattisgarh,0.775416,0.770042,28,approximate (area-weighted average of district...
7,dadra_nagar_haveli,0.027967,0.034276,3,approximate (area-weighted average of district...
8,delhi,0.099723,0.057043,11,approximate (area-weighted average of district...
9,goa,0.059485,0.058980,2,approximate (area-weighted average of district...


## Sanity check

A large gap between `crop_entropy_state` and `crop_entropy_district_avg` means districts within that state have quite different crop diversity from one another — the state number isn't just "the typical district," it reflects the combined variety across all of them (or, for the approximate method, is being pulled toward the larger districts).

In [6]:
state_df['state_minus_district_avg'] = state_df['crop_entropy_state'] - state_df['crop_entropy_district_avg']
state_df.sort_values('state_minus_district_avg', ascending=False)

,state,crop_entropy_state,crop_entropy_district_avg,n_districts,method,state_minus_district_avg
17,maharashtra,0.854009,0.755941,36,approximate (area-weighted average of district...,0.098068
24,puducherry,0.597998,0.500457,4,approximate (area-weighted average of district...,0.097541
11,haryana,0.585940,0.517332,22,approximate (area-weighted average of district...,0.068608
33,west_bengal,0.751209,0.690620,23,approximate (area-weighted average of district...,0.060590
21,mp,0.871765,0.827341,53,approximate (area-weighted average of district...,0.044424
8,delhi,0.099723,0.057043,11,approximate (area-weighted average of district...,0.042680
29,telangana,0.994612,0.964810,33,approximate (area-weighted average of district...,0.029802
14,karnataka,0.553993,0.532858,30,approximate (area-weighted average of district...,0.021135
31,up,0.468466,0.447582,75,approximate (area-weighted average of district...,0.020884
23,odisha,0.754674,0.742285,30,approximate (area-weighted average of district...,0.012389


## Write output CSV

In [7]:
OUTPUT_CSV = 'crop_entropy_by_state.csv'
state_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(state_df)} state rows to {OUTPUT_CSV}")

Wrote 34 state rows to crop_entropy_by_state.csv
